# FASE4-P4: Pipeline de Risco Pre-Entrega

**Objetivo:** Baseline logistico + XGBoost + SHAP + curva PR + score por vendedor  
**Referencia:** `docs/metrics_agreement.md` e `docs/feature_contract.md`  
**Ancora temporal:** `order_approved_at` — nenhuma feature pos-entrega no modelo

## (1) Load & Feature Matrix

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    precision_recall_curve,
)
from xgboost import XGBClassifier

sys.path.insert(0, "..")  # permite 'from src.features import ...' ao rodar do notebooks/
from src.features import PRE_DELIVERY_FEATURES, FORBIDDEN_FEATURES, TARGET_COLUMN

print(f"PRE_DELIVERY_FEATURES ({len(PRE_DELIVERY_FEATURES)} colunas): {PRE_DELIVERY_FEATURES}")
print(f"FORBIDDEN_FEATURES ({len(FORBIDDEN_FEATURES)} colunas): {FORBIDDEN_FEATURES}")
print(f"TARGET_COLUMN: {TARGET_COLUMN}")

PRE_DELIVERY_FEATURES (13 colunas): ['freight_value', 'price', 'freight_ratio', 'estimated_delivery_days', 'seller_state', 'customer_state', 'seller_customer_distance_km', 'product_weight_g', 'product_volume_cm3', 'product_category_name_english', 'order_item_count', 'payment_type', 'payment_installments']
FORBIDDEN_FEATURES (6 colunas): ['order_delivered_customer_date', 'review_score', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp', 'order_delivered_carrier_date']
TARGET_COLUMN: bad_review


C:\Users\Wey\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Verificacao anti-leakage (FALHA RAPIDO se contrato violado)
leakage = [c for c in FORBIDDEN_FEATURES if c in PRE_DELIVERY_FEATURES]
assert not leakage, f"LEAKAGE DETECTADO: {leakage}"
print("OK: nenhum leakage em PRE_DELIVERY_FEATURES")

OK: nenhum leakage em PRE_DELIVERY_FEATURES


In [3]:
df = pd.read_parquet("../data/gold/olist_gold.parquet")
print(f"Gold table: {len(df):,} linhas x {df.shape[1]} colunas")

missing = [c for c in PRE_DELIVERY_FEATURES if c not in df.columns]
assert not missing, f"Features ausentes na gold table: {missing}"
print(f"OK: todas as {len(PRE_DELIVERY_FEATURES)} features pre-entrega presentes")

Gold table: 97,456 linhas x 43 colunas
OK: todas as 13 features pre-entrega presentes


In [4]:
X = df[PRE_DELIVERY_FEATURES].copy()
y = df[TARGET_COLUMN].copy()

print(f"Dataset: {len(X):,} pedidos")
print(f"Classe positiva (bad_review=1): {y.mean():.1%}")
print(f"Features: {X.shape[1]} colunas")
print(f"\nNulos por feature:")
nulos = X.isnull().sum()[X.isnull().sum() > 0]
print(nulos if len(nulos) > 0 else "(nenhum nulo nas features)")

Dataset: 97,456 pedidos
Classe positiva (bad_review=1): 13.9%
Features: 13 colunas

Nulos por feature:
freight_value                       3
price                               3
freight_ratio                       4
seller_state                        3
seller_customer_distance_km       490
product_weight_g                   19
product_volume_cm3                 19
product_category_name_english    1412
order_item_count                    3
payment_type                        1
payment_installments                1
dtype: int64


In [5]:
NUMERIC_FEATURES = [
    "freight_value", "price", "freight_ratio", "estimated_delivery_days",
    "seller_customer_distance_km", "product_weight_g", "product_volume_cm3",
    "order_item_count", "payment_installments",
]
CATEGORICAL_FEATURES = [
    "seller_state", "customer_state",
    "product_category_name_english", "payment_type",
]

assert set(NUMERIC_FEATURES + CATEGORICAL_FEATURES) == set(PRE_DELIVERY_FEATURES), \
    "NUMERIC + CATEGORICAL nao cobrem exatamente PRE_DELIVERY_FEATURES"

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)
print(f"Treino: {len(X_train):,} | Teste: {len(X_test):,}")
print(f"Positivos treino: {y_train.mean():.1%} | Positivos teste: {y_test.mean():.1%}")

Treino: 77,964 | Teste: 19,492
Positivos treino: 13.9% | Positivos teste: 13.9%


## (2) Baseline — Logistic Regression

In [6]:
# Pipelines de pre-processamento com imputer para tratar nulos (Rule 2 auto-fix)
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, NUMERIC_FEATURES),
    ("cat", categorical_transformer, CATEGORICAL_FEATURES),
])

baseline_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",  # LOCKED: sem SMOTE (decisao CONTEXT.md)
        max_iter=1000,
        random_state=42,
    )),
])

baseline_pipeline.fit(X_train, y_train)
print("Baseline pipeline treinado.")

Baseline pipeline treinado.


In [7]:
def eval_model(pipeline, X_test, y_test, name):
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    y_pred = pipeline.predict(X_test)
    pr_auc = average_precision_score(y_test, y_proba)
    print(f"\n{'='*40}")
    print(f"{name} | PR-AUC: {pr_auc:.4f}")
    print(classification_report(y_test, y_pred, target_names=["good_review", "bad_review"]))
    return pr_auc

baseline_pr_auc = eval_model(baseline_pipeline, X_test, y_test, "Baseline LogReg")


Baseline LogReg | PR-AUC: 0.2207


              precision    recall  f1-score   support

 good_review       0.90      0.66      0.76     16788
  bad_review       0.20      0.53      0.29      2704

    accuracy                           0.64     19492
   macro avg       0.55      0.59      0.53     19492
weighted avg       0.80      0.64      0.70     19492



In [8]:
import os
os.makedirs("../models", exist_ok=True)

joblib.dump(baseline_pipeline, "../models/baseline_logreg.joblib")
print("Salvo: models/baseline_logreg.joblib")

# Round-trip verification
loaded_baseline = joblib.load("../models/baseline_logreg.joblib")
sample_scores = loaded_baseline.predict_proba(X_test.head(3))[:, 1]
print(f"Round-trip OK: {sample_scores.round(3)}")

Salvo: models/baseline_logreg.joblib
Round-trip OK: [0.533 0.629 0.646]


## (3) XGBoost

In [9]:
# scale_pos_weight DEVE ser calculado em y_train -- nao em y completo
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight: {scale_pos_weight:.2f} (neg={neg:,}, pos={pos:,})")

final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),  # mesmo ColumnTransformer do baseline (ja fitted em X_train)
    ("classifier", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=scale_pos_weight,
        eval_metric="aucpr",
        random_state=42,
        n_jobs=-1,
    )),
])

final_pipeline.fit(X_train, y_train)
print("XGBoost pipeline treinado.")

scale_pos_weight: 6.21 (neg=67,147, pos=10,817)


XGBoost pipeline treinado.


In [10]:
final_pr_auc = eval_model(final_pipeline, X_test, y_test, "XGBoost Final")

# Garantia de que o XGBoost supera o baseline
assert final_pr_auc > baseline_pr_auc, \
    f"XGBoost ({final_pr_auc:.4f}) deve superar baseline ({baseline_pr_auc:.4f})"
print(f"\nMelhora sobre baseline: +{final_pr_auc - baseline_pr_auc:.4f} PR-AUC")


XGBoost Final | PR-AUC: 0.2283
              precision    recall  f1-score   support

 good_review       0.90      0.70      0.79     16788
  bad_review       0.22      0.51      0.30      2704

    accuracy                           0.67     19492
   macro avg       0.56      0.60      0.55     19492
weighted avg       0.80      0.67      0.72     19492


Melhora sobre baseline: +0.0076 PR-AUC


In [11]:
import os
os.makedirs("../models", exist_ok=True)

# Serializar o Pipeline COMPLETO (preprocessor + estimador) -- nao apenas o XGBClassifier
joblib.dump(final_pipeline, "../models/final_pipeline.joblib")
print("Salvo: models/final_pipeline.joblib")

# Round-trip verification -- critico para garantir que o Streamlit conseguira carregar
loaded_final = joblib.load("../models/final_pipeline.joblib")
sample_scores = loaded_final.predict_proba(X_test.head(3))[:, 1]
print(f"Round-trip OK: {sample_scores.round(3)}")

Salvo: models/final_pipeline.joblib
Round-trip OK: [0.452 0.604 0.617]


## (4) SHAP

In [12]:
# Extrair classificador e transformar dados -- OBRIGATORIO antes do SHAP
xgb_model = final_pipeline.named_steps["classifier"]
X_test_transformed = final_pipeline.named_steps["preprocessor"].transform(X_test)

# Nomes de features apos OneHotEncoder
ohe = final_pipeline.named_steps["preprocessor"].named_transformers_["cat"]
feature_names = NUMERIC_FEATURES + list(ohe.get_feature_names_out(CATEGORICAL_FEATURES))

print(f"Features transformadas: {len(feature_names)}")
print(f"Dimensao X_test_transformed: {X_test_transformed.shape}")

Features transformadas: 134
Dimensao X_test_transformed: (19492, 134)


In [13]:
# Amostra maxima de 5000 -- TreeExplainer pode levar 10+ min em 100k linhas
# 5000 amostras sao suficientes para um beeswarm estavel (padrao de importancia converge acima de 2k)
sample_size = min(5000, len(X_test_transformed))
X_shap = X_test_transformed[:sample_size]

print(f"SHAP em {sample_size:,} amostras...")
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_shap)
print(f"SHAP calculado. Shape: {shap_values.shape}")

SHAP em 5,000 amostras...


SHAP calculado. Shape: (5000, 134)


In [14]:
import os
os.makedirs("../reports/figures", exist_ok=True)

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_shap,
    feature_names=feature_names,
    max_display=15,
    show=False,  # CRITICO: show=False para salvar em arquivo sem abrir janela
)
plt.title("SHAP -- Top 15 Features: Pre-Delivery Risk Model", fontsize=13, pad=12)
plt.tight_layout()
plt.savefig("../reports/figures/shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.close()
print("Salvo: reports/figures/shap_beeswarm.png")

Salvo: reports/figures/shap_beeswarm.png


In [15]:
import pandas as pd
import numpy as np

mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_df = (
    pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs_shap})
    .sort_values("mean_abs_shap", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
print("Top 15 features por importancia SHAP (mean |SHAP value|):")
print(shap_df.to_string(index=False))

Top 15 features por importancia SHAP (mean |SHAP value|):
                                            feature  mean_abs_shap
                                   order_item_count       0.188012
                                  customer_state_RJ       0.101439
                        seller_customer_distance_km       0.097569
                                              price       0.069036
                                    seller_state_SP       0.049435
                            estimated_delivery_days       0.044455
                                 product_volume_cm3       0.041082
                                   product_weight_g       0.040424
       product_category_name_english_bed_bath_table       0.035013
                                  customer_state_SP       0.027687
                                      freight_value       0.026339
product_category_name_english_computers_accessories       0.024623
                               payment_installments       0.017700
    

## (5) Threshold + Estimativa Operacional

In [16]:
# PLACEHOLDER — implementado no Plano 04-03
pass

## (6) Tabela de Vendedores

In [17]:
# PLACEHOLDER — implementado no Plano 04-03
pass

## (7) Serializar .joblib

In [18]:
# PLACEHOLDER — final_pipeline.joblib serializado no Plano 04-03
pass